In [1]:
import kagglehub
import tensorflow as tf

/home/zero/venvs/ds-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1788486085.656314    4099 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
print(tf.config.list_physical_devices('GPU'))
kagglehub.login()

# Download latest version
path = kagglehub.competition_download('rsna-knee-abnormality-detection')

print("Path to competition files:", path)

In [5]:
from helper_functions import *
study_ids = os.listdir("train_series")

train_df = pd.read_csv("train.csv")
labeled_data = train_df.dropna()

ids_for_studies_not_downloaded = labeled_data[~labeled_data["StudyInstanceUID"].isin(study_ids)]
ids_for_studies_not_downloaded.drop(columns=target_columns).drop(columns=["Report"]).to_csv("/home/zero/ds-projects/RSNA_KneeAbnormalityDetection/missing.csv", index=False)
ids_for_studies_not_downloaded

,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture


In [3]:
"train_images" in study_ids

True

In [ ]:
import pandas as pd
import os
from kaggle.api.kaggle_api_extended import KaggleApi

# 1. Authenticate with standard Kaggle API
api = KaggleApi()
api.authenticate()

# 2. Identify missing files
df = pd.read_csv('train.csv')
competition_name = 'rsna-knee-abnormality-detection'

# 3. Iterate and download missing files
for index, row in df.iterrows():
    # Construct the expected path based on your CSV structure
    study_id = row['StudyInstanceUID']
    expected_file = os.path.join('train_series', study_id)

    if not os.path.exists(expected_file):
        print(f"Missing {study_id}, downloading...")
        # Download the specific file or folder
        # Path inside the competition backend usually matches the zip structure
        api.competition_download_file(
            competition_name,
            f"train_images/{study_id}",
            path='train_series'
        )

In [4]:
import os
import zipfile

zip_path = "/home/zero/ds-projects/RSNA_KneeAbnormalityDetection/missing_labeled_data_part1.zip"
target_path = "train_series/"

os.makedirs(target_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(target_path)

In [4]:
import os
import shutil

base_dir = "train_series"
nested_dir = os.path.join(base_dir, "train_images")

if os.path.exists(nested_dir):
    # Move each folder/file from the nested directory to the base directory
    for item in os.listdir(nested_dir):
        source_path = os.path.join(nested_dir, item)
        destination_path = os.path.join(base_dir, item)

        shutil.move(source_path, destination_path)

    # Remove the now-empty extra 'train_series' directory
    os.rmdir(nested_dir)
    print("Folder structure fixed successfully.")
else:
    print(f"The directory '{nested_dir}' does not exist.")

Folder structure fixed successfully.


In [7]:
import os

print(os.path.abspath(os.getcwd()))

/root/.cache/kagglehub/competitions/rsna-knee-abnormality-detection
